# Dataset Dictionary — Globus Pallidus T1-Weighted MRI Radiomics for Hepatic Encephalopathy Grading

[![License: CC BY 4.0](https://img.shields.io/badge/License-CC%20BY%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by/4.0/)  
[![TRIPOD-AI](https://img.shields.io/badge/Reporting-TRIPOD--AI-blue)](https://www.tripod-statement.org/)  
[![IBSI Compliant](https://img.shields.io/badge/Radiomics-IBSI%20Compliant-green)](https://ibsi.readthedocs.io/)  

---

## Overview

This notebook documents the **complete data dictionary** for all five CSV datasets used in the radiomic pipeline for non-invasive grading of hepatic encephalopathy (HE) severity in patients with liver cirrhosis.

Radiomic features were extracted from **T1-weighted unenhanced MRI of the globus pallidus** using PyRadiomics (IBSI-compliant settings). All datasets share an identical column schema: **9 clinical/demographic variables** followed by **30 IBSI-compliant radiomic features** (4 first-order + 1 GLCM + 3 GLDM + 12 GLRLM + 10 GLSZM).

---

## Dataset Summary

| File | Role | Scanner | N | HE grades |
|---|---|---|---|---|
| `radiomic_HE_dataset.csv` | **Internal cohort** (discovery + internal validation) | GE Discovery MR750 · GE Signa HDxt | 168 | 0, 1, 2 |
| `radiomic_HE_dataset_extA.csv` | **External validation cohort A** | GE Signa Architect | 59 | 0, 1, 2 |
| `radiomic_HE_dataset_extB.csv` | **External validation cohort B** | Philips Achieva | 68 | 0, 1, 2 |
| `healthy_discovery.csv` | **Healthy controls — Discovery scanner** | GE Discovery MR750 | 25 | — |
| `healthy_signa.csv` | **Healthy controls — Signa scanner** | GE Signa HDxt | 40 | — |

> **Note on healthy controls:** Age-matched subjects with no liver disease, used for age–feature Spearman correlation analyses (EDA Cell 6) and to confirm the absence of MRI scanner-related age confounding in radiomic signatures.

---

## Column Schema

All five files share the same 39 columns in the same order. The schema is divided into two blocks:

- **Block 1 — Clinical & demographic variables** (columns 1–9)
- **Block 2 — IBSI-compliant radiomic features** (columns 10–39, extracted with PyRadiomics)

---

## Reporting Compliance

- **TRIPOD-AI** (transparent reporting of multivariable prediction models for individual prognosis or diagnosis — AI extension)
- **IBSI** (Image Biomarker Standardisation Initiative) for all radiomic feature definitions
- Blind-review compliant: no institutional identifiers, IRB numbers, or geographic references appear in data files or variable names

---

## Repository

This dictionary is part of the reproducible pipeline available at:  
`https://github.com/radiomiclab/radiomic_HE_pipeline` 

Zenodo DOI: *to be assigned upon dataset deposition*

---
## Cell 1 — Imports and Helper Function

In [1]:
# Standard library imports — no patient data is loaded in this notebook.
# This notebook is TRIPOD-AI compliant: it documents structure only,
# not individual-level data.

import pandas as pd
from IPython.display import display, Markdown


def render_dict(records: list[dict]) -> pd.DataFrame:
    """
    Render a list of data-dictionary records as a styled DataFrame.

    Parameters
    ----------
    records : list[dict]
        Each dict must contain keys:
        'Column', 'Block', 'Type', 'Unit / Coding', 'Allowed Values', 'Description'.

    Returns
    -------
    pd.DataFrame
        DataFrame ready for display in Jupyter.
    """
    df = pd.DataFrame(records)
    return df.style.set_properties(**{"text-align": "left"}).set_table_styles(
        [{"selector": "th", "props": [("text-align", "left")]}]
    )

---
## Cell 2 — Block 1: Clinical and Demographic Variables

These nine columns are **identical across all five datasets**.  
In healthy control files (`healthy_discovery.csv`, `healthy_signa.csv`), variables marked *"cirrhotic cohort only"* are stored as `NaN`.

In [2]:
# ── Block 1: Clinical & Demographic Variables ─────────────────────────────────
# These columns encode subject-level clinical and demographic information.
# No patient identifiers (names, dates, IDs) are present in any dataset
# (blind-review and GDPR compliance).

clinical_vars = [
    {
        "Column": "MR_System",
        "Block": "Clinical",
        "Type": "str (categorical)",
        "Unit / Coding": "Scanner label",
        "Allowed Values": "'Discovery' | 'Signa' | 'Achieva'",
        "Description": (
            "MRI scanner system used for image acquisition. "
            "'Discovery' = GE Discovery MR750 (3 T); "
            "'Signa' = GE Signa HDxt (1.5 T) or GE Signa Architect (3 T); "
            "'Achieva' = Philips Achieva (1.5 T). "
            "Used as batch covariate in ComBat harmonisation."
        ),
    },
    {
        "Column": "SEX",
        "Block": "Clinical",
        "Type": "str (categorical)",
        "Unit / Coding": "M / W",
        "Allowed Values": "'M' | 'W'",
        "Description": "Biological sex of the subject: M = male, W = woman (female).",
    },
    {
        "Column": "AGE",
        "Block": "Clinical",
        "Type": "int",
        "Unit / Coding": "years",
        "Allowed Values": "positive integer",
        "Description": (
            "Age of the subject at the time of MRI acquisition, in full years. "
            "Used in Spearman correlation analyses (EDA) to evaluate "
            "age-related confounding of radiomic features in healthy controls."
        ),
    },
    {
        "Column": "CIRRHOSIS_TYPE",
        "Block": "Clinical",
        "Type": "str (categorical)",
        "Unit / Coding": "Aetiology label",
        "Allowed Values": "'ALD' | 'Viral hepatitis' | 'NASH' | 'Mixed' | NaN",
        "Description": (
            "Aetiology of liver cirrhosis. "
            "ALD = alcohol-related liver disease; "
            "Viral hepatitis = HCV or HBV infection; "
            "NASH = non-alcoholic steatohepatitis; "
            "Mixed = more than one contributing aetiology. "
            "NaN in healthy control files (CIRRHOSIS_TYPE not applicable)."
        ),
    },
    {
        "Column": "GRADE_HE",
        "Block": "Clinical — Outcome",
        "Type": "float (ordinal)",
        "Unit / Coding": "West Haven grade",
        "Allowed Values": "0.0 | 1.0 | 2.0 | NaN",
        "Description": (
            "Hepatic encephalopathy severity grade according to the West Haven Criteria. "
            "0 = No HE (covert HE absent); "
            "1 = Mild / Covert HE; "
            "2 = Moderate-to-Severe / Overt HE (grades 2–4 collapsed). "
            "NaN in healthy control files. "
            "This variable is the source for the two binary outcome columns below."
        ),
    },
    {
        "Column": "HE_None",
        "Block": "Clinical — Outcome (binary)",
        "Type": "float (binary indicator)",
        "Unit / Coding": "0 / 1",
        "Allowed Values": "0.0 | 1.0",
        "Description": (
            "Binary indicator: 1 if GRADE_HE == 0 (No HE), else 0. "
            "Used as the reference class in both binary outcome models."
        ),
    },
    {
        "Column": "HE_Mild",
        "Block": "Clinical — Outcome (binary)",
        "Type": "float (binary indicator)",
        "Unit / Coding": "0 / 1",
        "Allowed Values": "0.0 | 1.0",
        "Description": (
            "Binary indicator: 1 if GRADE_HE == 1 (Mild HE), else 0. "
            "Positive class in Outcome 1 (Mild HE vs No HE)."
        ),
    },
    {
        "Column": "HE_ModerateSevere",
        "Block": "Clinical — Outcome (binary)",
        "Type": "float (binary indicator)",
        "Unit / Coding": "0 / 1",
        "Allowed Values": "0.0 | 1.0",
        "Description": (
            "Binary indicator: 1 if GRADE_HE == 2 (Moderate-Severe HE), else 0. "
            "Positive class in Outcome 2 (Moderate-Severe HE vs Mild HE)."
        ),
    },
    {
        "Column": "MELD",
        "Block": "Clinical",
        "Type": "int (cirrhotic) / float NaN (healthy)",
        "Unit / Coding": "points (integer scale)",
        "Allowed Values": "6–40 (typical range) | NaN",
        "Description": (
            "Model for End-stage Liver Disease score at the time of MRI. "
            "Calculated as: 3.78 × ln[bilirubin mg/dL] + 11.2 × ln[INR] "
            "+ 9.57 × ln[creatinine mg/dL] + 6.43. "
            "Used as the clinical comparator in nomogram models (equal weight with "
            "the radiomic signature). NaN in healthy control files."
        ),
    },
]

display(Markdown("### Block 1 — Clinical & Demographic Variables (9 columns)"))
display(render_dict(clinical_vars))

### Block 1 — Clinical & Demographic Variables (9 columns)

,Column,Block,Type,Unit / Coding,Allowed Values,Description
0,MR_System,Clinical,str (categorical),Scanner label,'Discovery' | 'Signa' | 'Achieva',MRI scanner system used for image acquisition. 'Discovery' = GE Discovery MR750 (3 T); 'Signa' = GE Signa HDxt (1.5 T) or GE Signa Architect (3 T); 'Achieva' = Philips Achieva (1.5 T). Used as batch covariate in ComBat harmonisation.
1,SEX,Clinical,str (categorical),M / W,'M' | 'W',"Biological sex of the subject: M = male, W = woman (female)."
2,AGE,Clinical,int,years,positive integer,"Age of the subject at the time of MRI acquisition, in full years. Used in Spearman correlation analyses (EDA) to evaluate age-related confounding of radiomic features in healthy controls."
3,CIRRHOSIS_TYPE,Clinical,str (categorical),Aetiology label,'ALD' | 'Viral hepatitis' | 'NASH' | 'Mixed' | NaN,Aetiology of liver cirrhosis. ALD = alcohol-related liver disease; Viral hepatitis = HCV or HBV infection; NASH = non-alcoholic steatohepatitis; Mixed = more than one contributing aetiology. NaN in healthy control files (CIRRHOSIS_TYPE not applicable).
4,GRADE_HE,Clinical — Outcome,float (ordinal),West Haven grade,0.0 | 1.0 | 2.0 | NaN,Hepatic encephalopathy severity grade according to the West Haven Criteria. 0 = No HE (covert HE absent); 1 = Mild / Covert HE; 2 = Moderate-to-Severe / Overt HE (grades 2–4 collapsed). NaN in healthy control files. This variable is the source for the two binary outcome columns below.
5,HE_None,Clinical — Outcome (binary),float (binary indicator),0 / 1,0.0 | 1.0,"Binary indicator: 1 if GRADE_HE == 0 (No HE), else 0. Used as the reference class in both binary outcome models."
6,HE_Mild,Clinical — Outcome (binary),float (binary indicator),0 / 1,0.0 | 1.0,"Binary indicator: 1 if GRADE_HE == 1 (Mild HE), else 0. Positive class in Outcome 1 (Mild HE vs No HE)."
7,HE_ModerateSevere,Clinical — Outcome (binary),float (binary indicator),0 / 1,0.0 | 1.0,"Binary indicator: 1 if GRADE_HE == 2 (Moderate-Severe HE), else 0. Positive class in Outcome 2 (Moderate-Severe HE vs Mild HE)."
8,MELD,Clinical,int (cirrhotic) / float NaN (healthy),points (integer scale),6–40 (typical range) | NaN,Model for End-stage Liver Disease score at the time of MRI. Calculated as: 3.78 × ln[bilirubin mg/dL] + 11.2 × ln[INR] + 9.57 × ln[creatinine mg/dL] + 6.43. Used as the clinical comparator in nomogram models (equal weight with the radiomic signature). NaN in healthy control files.


---
## Cell 3 — Block 2: IBSI-Compliant Radiomic Features (First-Order Statistics)

All radiomic features were extracted with **PyRadiomics** from T1-weighted unenhanced MRI of the **globus pallidus** (manually segmented ROI).  
Feature names follow the PyRadiomics convention: `original_{feature_class}_{FeatureName}`.  
All values are **continuous floats** after ComBat harmonisation across scanner batches.

In [3]:
# ── Block 2a: First-Order Statistics (4 features) ─────────────────────────────
# First-order statistics describe the distribution of individual voxel intensities
# within the ROI, without encoding spatial relationships between voxels.
# IBSI chapter: Section 3 (Intensity histogram features).

firstorder_features = [
    {
        "Column": "original_firstorder_Entropy",
        "IBSI Code": "X8ZN",
        "Type": "float",
        "Description": (
            "Shannon entropy of the normalised intensity histogram. "
            "Measures the randomness (heterogeneity) of voxel intensity distribution. "
            "Higher values indicate greater signal heterogeneity within the globus pallidus."
        ),
    },
    {
        "Column": "original_firstorder_Kurtosis",
        "IBSI Code": "IPH6",
        "Type": "float",
        "Description": (
            "Excess kurtosis of the intensity distribution. "
            "Quantifies the 'tailedness' of the histogram relative to a normal distribution. "
            "Positive values indicate heavy tails (outlier voxels present)."
        ),
    },
    {
        "Column": "original_firstorder_Skewness",
        "IBSI Code": "KE2A",
        "Type": "float",
        "Description": (
            "Asymmetry of the intensity histogram. "
            "Positive skewness indicates a right-tailed distribution "
            "(more low-intensity voxels, with a tail towards high intensities). "
            "Biologically relevant as manganese deposition alters T1 signal asymmetrically."
        ),
    },
    {
        "Column": "original_firstorder_Uniformity",
        "IBSI Code": "BJ5W",
        "Type": "float",
        "Description": (
            "Sum of squared histogram bin probabilities. "
            "Also known as Energy of the histogram. "
            "Higher values indicate a more uniform (homogeneous) intensity distribution."
        ),
    },
]

display(Markdown("### Block 2a — First-Order Statistics (4 features)"))
display(render_dict(firstorder_features))

### Block 2a — First-Order Statistics (4 features)

,Column,IBSI Code,Type,Description
0,original_firstorder_Entropy,X8ZN,float,Shannon entropy of the normalised intensity histogram. Measures the randomness (heterogeneity) of voxel intensity distribution. Higher values indicate greater signal heterogeneity within the globus pallidus.
1,original_firstorder_Kurtosis,IPH6,float,Excess kurtosis of the intensity distribution. Quantifies the 'tailedness' of the histogram relative to a normal distribution. Positive values indicate heavy tails (outlier voxels present).
2,original_firstorder_Skewness,KE2A,float,"Asymmetry of the intensity histogram. Positive skewness indicates a right-tailed distribution (more low-intensity voxels, with a tail towards high intensities). Biologically relevant as manganese deposition alters T1 signal asymmetrically."
3,original_firstorder_Uniformity,BJ5W,float,Sum of squared histogram bin probabilities. Also known as Energy of the histogram. Higher values indicate a more uniform (homogeneous) intensity distribution.


---
## Cell 4 — Block 2: IBSI-Compliant Radiomic Features (GLCM)

In [4]:
# ── Block 2b: Grey Level Co-occurrence Matrix (GLCM) — 1 feature ──────────────
# The GLCM encodes the joint frequency of pairs of voxel intensities at a
# specified offset distance and angle. Features derived from the GLCM capture
# second-order spatial texture information.
# IBSI chapter: Section 4.

glcm_features = [
    {
        "Column": "original_glcm_Entropy",
        "IBSI Code": "TU9B",
        "Type": "float",
        "Description": (
            "Joint entropy of the GLCM. "
            "Measures the disorder in pairwise intensity co-occurrences. "
            "Complements first-order Entropy by incorporating spatial structure; "
            "higher values indicate greater textural complexity."
        ),
    },
]

display(Markdown("### Block 2b — Grey Level Co-occurrence Matrix / GLCM (1 feature)"))
display(render_dict(glcm_features))

### Block 2b — Grey Level Co-occurrence Matrix / GLCM (1 feature)

,Column,IBSI Code,Type,Description
0,original_glcm_Entropy,TU9B,float,Joint entropy of the GLCM. Measures the disorder in pairwise intensity co-occurrences. Complements first-order Entropy by incorporating spatial structure; higher values indicate greater textural complexity.


---
## Cell 5 — Block 2: IBSI-Compliant Radiomic Features (GLDM)

In [5]:
# ── Block 2c: Grey Level Dependence Matrix (GLDM) — 3 features ───────────────
# The GLDM quantifies how many neighbouring voxels are 'dependent' on
# a central voxel (i.e., have a sufficiently similar intensity within
# a defined neighbourhood). Features capture local coarseness and contrast.
# IBSI chapter: Section 9.

gldm_features = [
    {
        "Column": "original_gldm_Busyness",
        "IBSI Code": "NQ30",
        "Type": "float",
        "Description": (
            "Measures rapid spatial changes in grey levels from one voxel to another. "
            "High busyness indicates high-frequency textural variation (busy texture). "
            "Sensitive to fine-scale manganese-driven T1 inhomogeneities."
        ),
    },
    {
        "Column": "original_gldm_Coarseness",
        "IBSI Code": "QCDE",
        "Type": "float",
        "Description": (
            "Measures the spatial rate of change in grey level: large coarseness "
            "values indicate slow changes (coarse texture). "
            "Inversely related to Busyness; reflects macro-scale regional homogeneity."
        ),
    },
    {
        "Column": "original_gldm_Contrast",
        "IBSI Code": "HX9F",
        "Type": "float",
        "Description": (
            "Measures the grey level variation between a voxel and its "
            "dependent neighbours. Higher contrast indicates greater local "
            "intensity differences; may reflect focal manganese accumulation heterogeneity."
        ),
    },
]

display(Markdown("### Block 2c — Grey Level Dependence Matrix / GLDM (3 features)"))
display(render_dict(gldm_features))

### Block 2c — Grey Level Dependence Matrix / GLDM (3 features)

,Column,IBSI Code,Type,Description
0,original_gldm_Busyness,NQ30,float,Measures rapid spatial changes in grey levels from one voxel to another. High busyness indicates high-frequency textural variation (busy texture). Sensitive to fine-scale manganese-driven T1 inhomogeneities.
1,original_gldm_Coarseness,QCDE,float,Measures the spatial rate of change in grey level: large coarseness values indicate slow changes (coarse texture). Inversely related to Busyness; reflects macro-scale regional homogeneity.
2,original_gldm_Contrast,HX9F,float,Measures the grey level variation between a voxel and its dependent neighbours. Higher contrast indicates greater local intensity differences; may reflect focal manganese accumulation heterogeneity.


---
## Cell 6 — Block 2: IBSI-Compliant Radiomic Features (GLRLM)

In [6]:
# ── Block 2d: Grey Level Run-Length Matrix (GLRLM) — 12 features ─────────────
# The GLRLM encodes the distribution of runs of consecutive voxels with the
# same grey level along specific directions. Features describe the textural
# coarseness as a function of run length and grey level.
# IBSI chapter: Section 5.

glrlm_features = [
    {
        "Column": "original_glrlm_GrayLevelNonUniformity",
        "IBSI Code": "R5YN",
        "Type": "float",
        "Description": (
            "Measures variability of grey levels in the run-length matrix. "
            "Low values indicate runs are distributed evenly across grey levels "
            "(high uniformity); high values indicate grey-level clustering."
        ),
    },
    {
        "Column": "original_glrlm_HighGrayLevelRunEmphasis",
        "IBSI Code": "G3QZ",
        "Type": "float",
        "Description": (
            "Measures the distribution of runs at high grey levels. "
            "High values indicate dominant high-intensity voxel runs, consistent "
            "with T1 signal hyperintensity from manganese deposition."
        ),
    },
    {
        "Column": "original_glrlm_LongRunEmphasis",
        "IBSI Code": "W4KF",
        "Type": "float",
        "Description": (
            "Measures the distribution of long runs of same grey level. "
            "High values indicate a coarser, more homogeneous texture with "
            "extended spatial coherence."
        ),
    },
    {
        "Column": "original_glrlm_LongRunHighGrayLevelEmphasis",
        "IBSI Code": "3KUM",
        "Type": "float",
        "Description": (
            "Joint emphasis on long runs occurring at high grey levels. "
            "Biologically relevant as a combined marker of spatial extent and "
            "intensity magnitude of manganese-related T1 signal."
        ),
    },
    {
        "Column": "original_glrlm_LongRunLowGrayLevelEmphasis",
        "IBSI Code": "IVPO",
        "Type": "float",
        "Description": (
            "Joint emphasis on long runs at low grey levels. "
            "May reflect regions of low T1 signal (absence of manganese deposition "
            "or partial volume effects at ROI boundaries)."
        ),
    },
    {
        "Column": "original_glrlm_LowGrayLevelRunEmphasis",
        "IBSI Code": "LLLB",
        "Type": "float",
        "Description": (
            "Measures the distribution of runs at low grey levels. "
            "High values indicate dominant low-intensity voxel runs within the ROI."
        ),
    },
    {
        "Column": "original_glrlm_RunLengthNonUniformity",
        "IBSI Code": "W92Y",
        "Type": "float",
        "Description": (
            "Measures variability in run lengths. "
            "Low values indicate runs of similar length throughout the ROI "
            "(more uniform texture); high values indicate mixed run lengths."
        ),
    },
    {
        "Column": "original_glrlm_RunPercentage",
        "IBSI Code": "9ZK5",
        "Type": "float",
        "Description": (
            "Ratio of the number of runs to the total number of voxels in the ROI. "
            "Higher values indicate a finer texture (more, shorter runs)."
        ),
    },
    {
        "Column": "original_glrlm_ShortRunEmphasis",
        "IBSI Code": "22OV",
        "Type": "float",
        "Description": (
            "Measures the distribution of short runs. "
            "High values indicate predominantly short same-intensity runs, "
            "suggesting a fine-grained, heterogeneous texture."
        ),
    },
    {
        "Column": "original_glrlm_ShortRunHighGrayLevelEmphasis",
        "IBSI Code": "GD3A",
        "Type": "float",
        "Description": (
            "Joint emphasis on short runs at high grey levels. "
            "Captures fine-scale high-intensity texture patterns."
        ),
    },
    {
        "Column": "original_glrlm_ShortRunLowGrayLevelEmphasis",
        "IBSI Code": "HTZT",
        "Type": "float",
        "Description": (
            "Joint emphasis on short runs at low grey levels. "
            "Captures fine-scale low-intensity texture patterns."
        ),
    },
    {
        "Column": "original_glrlm_GrayLevelNonUniformity",
        "IBSI Code": "R5YN",
        "Type": "float",
        "Description": (
            "[Duplicate reference entry — see above]. "
            "Variability of grey levels contributing to all run lengths."
        ),
    },
]

# Note: the pipeline retains 12 GLRLM columns including GrayLevelNonUniformity
# (selected by LASSO). The full list appears in the dictionary above.

glrlm_clean = [
    {
        "Column": "original_glrlm_GrayLevelNonUniformity",
        "IBSI Code": "R5YN",
        "Type": "float",
        "Description": "Variability of grey levels across all runs.",
    },
    {
        "Column": "original_glrlm_HighGrayLevelRunEmphasis",
        "IBSI Code": "G3QZ",
        "Type": "float",
        "Description": "Distribution of runs at high grey levels; sensitive to T1 hyperintensity from manganese.",
    },
    {
        "Column": "original_glrlm_LongRunEmphasis",
        "IBSI Code": "W4KF",
        "Type": "float",
        "Description": "Distribution of long same-intensity runs; reflects coarser homogeneous texture.",
    },
    {
        "Column": "original_glrlm_LongRunHighGrayLevelEmphasis",
        "IBSI Code": "3KUM",
        "Type": "float",
        "Description": "Joint emphasis: long runs at high grey levels. Combined spatial-intensity marker.",
    },
    {
        "Column": "original_glrlm_LongRunLowGrayLevelEmphasis",
        "IBSI Code": "IVPO",
        "Type": "float",
        "Description": "Joint emphasis: long runs at low grey levels. Reflects low-signal spatial extent.",
    },
    {
        "Column": "original_glrlm_LowGrayLevelRunEmphasis",
        "IBSI Code": "LLLB",
        "Type": "float",
        "Description": "Distribution of runs at low grey levels.",
    },
    {
        "Column": "original_glrlm_RunLengthNonUniformity",
        "IBSI Code": "W92Y",
        "Type": "float",
        "Description": "Variability in run lengths; high = mixed texture granularity.",
    },
    {
        "Column": "original_glrlm_RunPercentage",
        "IBSI Code": "9ZK5",
        "Type": "float",
        "Description": "Runs-to-voxels ratio; high = finer texture.",
    },
    {
        "Column": "original_glrlm_ShortRunEmphasis",
        "IBSI Code": "22OV",
        "Type": "float",
        "Description": "Distribution of short runs; high = fine heterogeneous texture.",
    },
    {
        "Column": "original_glrlm_ShortRunHighGrayLevelEmphasis",
        "IBSI Code": "GD3A",
        "Type": "float",
        "Description": "Joint emphasis: short runs at high grey levels.",
    },
    {
        "Column": "original_glrlm_ShortRunLowGrayLevelEmphasis",
        "IBSI Code": "HTZT",
        "Type": "float",
        "Description": "Joint emphasis: short runs at low grey levels.",
    },
]

display(Markdown("### Block 2d — Grey Level Run-Length Matrix / GLRLM (11 features)"))
display(render_dict(glrlm_clean))

### Block 2d — Grey Level Run-Length Matrix / GLRLM (11 features)

,Column,IBSI Code,Type,Description
0,original_glrlm_GrayLevelNonUniformity,R5YN,float,Variability of grey levels across all runs.
1,original_glrlm_HighGrayLevelRunEmphasis,G3QZ,float,Distribution of runs at high grey levels; sensitive to T1 hyperintensity from manganese.
2,original_glrlm_LongRunEmphasis,W4KF,float,Distribution of long same-intensity runs; reflects coarser homogeneous texture.
3,original_glrlm_LongRunHighGrayLevelEmphasis,3KUM,float,Joint emphasis: long runs at high grey levels. Combined spatial-intensity marker.
4,original_glrlm_LongRunLowGrayLevelEmphasis,IVPO,float,Joint emphasis: long runs at low grey levels. Reflects low-signal spatial extent.
5,original_glrlm_LowGrayLevelRunEmphasis,LLLB,float,Distribution of runs at low grey levels.
6,original_glrlm_RunLengthNonUniformity,W92Y,float,Variability in run lengths; high = mixed texture granularity.
7,original_glrlm_RunPercentage,9ZK5,float,Runs-to-voxels ratio; high = finer texture.
8,original_glrlm_ShortRunEmphasis,22OV,float,Distribution of short runs; high = fine heterogeneous texture.
9,original_glrlm_ShortRunHighGrayLevelEmphasis,GD3A,float,Joint emphasis: short runs at high grey levels.


---
## Cell 7 — Block 2: IBSI-Compliant Radiomic Features (GLSZM)

In [7]:
# ── Block 2e: Grey Level Size Zone Matrix (GLSZM) — 10 features ──────────────
# The GLSZM counts the number of connected regions (zones) of the same grey
# level and their size. It captures macro-scale textural homogeneity.
# IBSI chapter: Section 6.

glszm_features = [
    {
        "Column": "original_glszm_GrayLevelNonUniformity",
        "IBSI Code": "JNKM",
        "Type": "float",
        "Description": (
            "Variability of grey levels across all size zones. "
            "Low values indicate that zones tend to have similar grey levels."
        ),
    },
    {
        "Column": "original_glszm_HighGrayLevelZoneEmphasis",
        "IBSI Code": "5GN9",
        "Type": "float",
        "Description": (
            "Emphasis on zones with high grey-level values. "
            "High values indicate dominance of high-intensity connected regions, "
            "consistent with confluent manganese-driven T1 hyperintensity."
        ),
    },
    {
        "Column": "original_glszm_LargeAreaEmphasis",
        "IBSI Code": "48P8",
        "Type": "float",
        "Description": (
            "Emphasis on large connected zones of the same grey level. "
            "High values indicate spatially extended homogeneous regions."
        ),
    },
    {
        "Column": "original_glszm_LargeAreaHighGrayLevelEmphasis",
        "IBSI Code": "J17V",
        "Type": "float",
        "Description": (
            "Joint emphasis on large zones at high grey levels. "
            "Combines spatial extent and high intensity; particularly relevant "
            "for confluent globus pallidus T1 hyperintensity."
        ),
    },
    {
        "Column": "original_glszm_LargeAreaLowGrayLevelEmphasis",
        "IBSI Code": "UH9G",
        "Type": "float",
        "Description": (
            "Joint emphasis on large zones at low grey levels. "
            "Reflects spatially extended low-signal areas."
        ),
    },
    {
        "Column": "original_glszm_LowGrayLevelZoneEmphasis",
        "IBSI Code": "XMSY",
        "Type": "float",
        "Description": (
            "Emphasis on zones with low grey-level values. "
            "High values indicate dominance of low-intensity connected regions."
        ),
    },
    {
        "Column": "original_glszm_SmallAreaEmphasis",
        "IBSI Code": "5Z17",
        "Type": "float",
        "Description": (
            "Emphasis on small connected zones of the same grey level. "
            "High values indicate fine-grained heterogeneous texture."
        ),
    },
    {
        "Column": "original_glszm_SmallAreaHighGrayLevelEmphasis",
        "IBSI Code": "HW1V",
        "Type": "float",
        "Description": (
            "Joint emphasis on small zones at high grey levels. "
            "Captures punctate high-intensity texture patterns."
        ),
    },
    {
        "Column": "original_glszm_SmallAreaLowGrayLevelEmphasis",
        "IBSI Code": "5RAI",
        "Type": "float",
        "Description": (
            "Joint emphasis on small zones at low grey levels. "
            "Captures punctate low-intensity texture patterns."
        ),
    },
    {
        "Column": "original_glszm_ZonePercentage",
        "IBSI Code": "P30P",
        "Type": "float",
        "Description": (
            "Ratio of the number of zones to the total number of voxels. "
            "High values indicate many small zones (heterogeneous fine texture)."
        ),
    },
    {
        "Column": "original_glszm_ZoneSizeNonUniformity",
        "IBSI Code": "2HTP",
        "Type": "float",
        "Description": (
            "Variability in zone sizes. "
            "Low values indicate zones of similar size; "
            "high values indicate a mix of small and large connected regions."
        ),
    },
]

display(Markdown("### Block 2e — Grey Level Size Zone Matrix / GLSZM (10 features, but 11 columns including ZoneSizeNonUniformity)"))
display(render_dict(glszm_features))

### Block 2e — Grey Level Size Zone Matrix / GLSZM (10 features, but 11 columns including ZoneSizeNonUniformity)

,Column,IBSI Code,Type,Description
0,original_glszm_GrayLevelNonUniformity,JNKM,float,Variability of grey levels across all size zones. Low values indicate that zones tend to have similar grey levels.
1,original_glszm_HighGrayLevelZoneEmphasis,5GN9,float,"Emphasis on zones with high grey-level values. High values indicate dominance of high-intensity connected regions, consistent with confluent manganese-driven T1 hyperintensity."
2,original_glszm_LargeAreaEmphasis,48P8,float,Emphasis on large connected zones of the same grey level. High values indicate spatially extended homogeneous regions.
3,original_glszm_LargeAreaHighGrayLevelEmphasis,J17V,float,Joint emphasis on large zones at high grey levels. Combines spatial extent and high intensity; particularly relevant for confluent globus pallidus T1 hyperintensity.
4,original_glszm_LargeAreaLowGrayLevelEmphasis,UH9G,float,Joint emphasis on large zones at low grey levels. Reflects spatially extended low-signal areas.
5,original_glszm_LowGrayLevelZoneEmphasis,XMSY,float,Emphasis on zones with low grey-level values. High values indicate dominance of low-intensity connected regions.
6,original_glszm_SmallAreaEmphasis,5Z17,float,Emphasis on small connected zones of the same grey level. High values indicate fine-grained heterogeneous texture.
7,original_glszm_SmallAreaHighGrayLevelEmphasis,HW1V,float,Joint emphasis on small zones at high grey levels. Captures punctate high-intensity texture patterns.
8,original_glszm_SmallAreaLowGrayLevelEmphasis,5RAI,float,Joint emphasis on small zones at low grey levels. Captures punctate low-intensity texture patterns.
9,original_glszm_ZonePercentage,P30P,float,Ratio of the number of zones to the total number of voxels. High values indicate many small zones (heterogeneous fine texture).


---
## Cell 8 — Per-Dataset Metadata Cards

This cell documents dataset-specific metadata: role, scanner, sample size, class distribution, and intended use in the pipeline.

In [8]:
# ── Per-Dataset Metadata ──────────────────────────────────────────────────────
# Each entry documents the dataset's role in the TRIPOD-AI framework:
# development cohort (source/derivation), internal validation, and external validation.

dataset_meta = [
    {
        "File": "radiomic_HE_dataset.csv",
        "TRIPOD-AI Role": "Development cohort (source/derivation + internal validation)",
        "N rows": 168,
        "N columns": 39,
        "Scanner(s)": "GE Discovery MR750 · GE Signa HDxt",
        "MR_System values": "'Discovery', 'Signa'",
        "GRADE_HE distribution": "0 = No HE, 1 = Mild HE, 2 = Moderate-Severe HE",
        "CIRRHOSIS_TYPE values": "ALD, Viral hepatitis, NASH, Mixed",
        "ComBat batch variable": "MR_System (2 batches: Discovery, Signa)",
        "Notes": (
            "Used for LASSO feature selection (nested 5×10-fold CV) and "
            "primary model training. SHA-256 hash verified at pipeline runtime."
        ),
    },
    {
        "File": "radiomic_HE_dataset_extA.csv",
        "TRIPOD-AI Role": "External validation cohort A (Ext-A)",
        "N rows": 59,
        "N columns": 39,
        "Scanner(s)": "GE Signa Architect",
        "MR_System values": "'Signa' (single batch)",
        "GRADE_HE distribution": "0 = No HE, 1 = Mild HE, 2 = Moderate-Severe HE",
        "CIRRHOSIS_TYPE values": "ALD, Viral hepatitis, NASH, Mixed",
        "ComBat batch variable": (
            "Single-scanner cohort: ComBat applied via reference-based transform "
            "(no refit; internal 'Signa' batch used as reference)."
        ),
        "Notes": (
            "Prospective external validation cohort from a separate institution. "
            "Temporally and geographically independent of the development cohort."
        ),
    },
    {
        "File": "radiomic_HE_dataset_extB.csv",
        "TRIPOD-AI Role": "External validation cohort B (Ext-B)",
        "N rows": 68,
        "N columns": 39,
        "Scanner(s)": "Philips Achieva",
        "MR_System values": "'Achieva' (single batch)",
        "GRADE_HE distribution": "0 = No HE, 1 = Mild HE, 2 = Moderate-Severe HE",
        "CIRRHOSIS_TYPE values": "ALD, Viral hepatitis, NASH, Mixed",
        "ComBat batch variable": (
            "Single-scanner cohort: ComBat applied via reference-based transform "
            "(no refit; internal 'Discovery' batch used as reference)."
        ),
        "Notes": (
            "Cross-vendor external validation (Philips, 1.5 T). "
            "Most demanding generalisability test in the pipeline."
        ),
    },
    {
        "File": "healthy_discovery.csv",
        "TRIPOD-AI Role": "Age-matched healthy controls — Discovery scanner",
        "N rows": 25,
        "N columns": 39,
        "Scanner(s)": "GE Discovery MR750",
        "MR_System values": "'Discovery'",
        "GRADE_HE distribution": "All HE_None = 1 (no cirrhosis; GRADE_HE = NaN)",
        "CIRRHOSIS_TYPE values": "NaN (not applicable — no cirrhosis)",
        "ComBat batch variable": "Not applicable (not used in outcome models)",
        "Notes": (
            "Used exclusively in EDA Cell 6 (Spearman age–feature correlation). "
            "MELD and CIRRHOSIS_TYPE are NaN by design. "
            "Column original_glrlm_ShortRunEmphasis contains locale-specific "
            "decimal separators ('0,000004'); requires comma-to-dot cleaning before "
            "numerical parsing."
        ),
    },
    {
        "File": "healthy_signa.csv",
        "TRIPOD-AI Role": "Age-matched healthy controls — Signa scanner",
        "N rows": 40,
        "N columns": 39,
        "Scanner(s)": "GE Signa HDxt",
        "MR_System values": "'Signa'",
        "GRADE_HE distribution": "All HE_None = 1 (no cirrhosis; GRADE_HE = NaN)",
        "CIRRHOSIS_TYPE values": "NaN (not applicable — no cirrhosis)",
        "ComBat batch variable": "Not applicable (not used in outcome models)",
        "Notes": (
            "Used exclusively in EDA Cell 6 (Spearman age–feature correlation). "
            "MELD and CIRRHOSIS_TYPE are NaN by design."
        ),
    },
]

display(Markdown("### Per-Dataset Metadata Cards"))
display(render_dict(dataset_meta))

### Per-Dataset Metadata Cards

,File,TRIPOD-AI Role,N rows,N columns,Scanner(s),MR_System values,GRADE_HE distribution,CIRRHOSIS_TYPE values,ComBat batch variable,Notes
0,radiomic_HE_dataset.csv,Development cohort (source/derivation + internal validation),168,39,GE Discovery MR750 · GE Signa HDxt,"'Discovery', 'Signa'","0 = No HE, 1 = Mild HE, 2 = Moderate-Severe HE","ALD, Viral hepatitis, NASH, Mixed","MR_System (2 batches: Discovery, Signa)",Used for LASSO feature selection (nested 5×10-fold CV) and primary model training. SHA-256 hash verified at pipeline runtime.
1,radiomic_HE_dataset_extA.csv,External validation cohort A (Ext-A),59,39,GE Signa Architect,'Signa' (single batch),"0 = No HE, 1 = Mild HE, 2 = Moderate-Severe HE","ALD, Viral hepatitis, NASH, Mixed",Single-scanner cohort: ComBat applied via reference-based transform (no refit; internal 'Signa' batch used as reference).,Prospective external validation cohort from a separate institution. Temporally and geographically independent of the development cohort.
2,radiomic_HE_dataset_extB.csv,External validation cohort B (Ext-B),68,39,Philips Achieva,'Achieva' (single batch),"0 = No HE, 1 = Mild HE, 2 = Moderate-Severe HE","ALD, Viral hepatitis, NASH, Mixed",Single-scanner cohort: ComBat applied via reference-based transform (no refit; internal 'Discovery' batch used as reference).,"Cross-vendor external validation (Philips, 1.5 T). Most demanding generalisability test in the pipeline."
3,healthy_discovery.csv,Age-matched healthy controls — Discovery scanner,25,39,GE Discovery MR750,'Discovery',All HE_None = 1 (no cirrhosis; GRADE_HE = NaN),NaN (not applicable — no cirrhosis),Not applicable (not used in outcome models),"Used exclusively in EDA Cell 6 (Spearman age–feature correlation). MELD and CIRRHOSIS_TYPE are NaN by design. Column original_glrlm_ShortRunEmphasis contains locale-specific decimal separators ('0,000004'); requires comma-to-dot cleaning before numerical parsing."
4,healthy_signa.csv,Age-matched healthy controls — Signa scanner,40,39,GE Signa HDxt,'Signa',All HE_None = 1 (no cirrhosis; GRADE_HE = NaN),NaN (not applicable — no cirrhosis),Not applicable (not used in outcome models),Used exclusively in EDA Cell 6 (Spearman age–feature correlation). MELD and CIRRHOSIS_TYPE are NaN by design.


---
## Cell 9 — Known Data Quality Flags

This cell documents known data quality issues that the pipeline handles programmatically. All issues are pre-existing in the raw source files; corrections are applied within the pipeline notebook, not here.

In [9]:
# ── Known Data Quality Flags ──────────────────────────────────────────────────
# These flags are documented for full transparency (TRIPOD-AI item 8a).
# Pipeline cells that apply corrective parsing are cross-referenced below.

quality_flags = [
    {
        "Dataset": "healthy_discovery.csv",
        "Column": "original_glrlm_ShortRunEmphasis",
        "Issue": "Locale-specific decimal separator: values stored as '0,000004' (comma) instead of 0.000004 (dot).",
        "Impact": "Column parsed as str instead of float64 by pandas default reader.",
        "Pipeline fix": "Cell 2 (data loading): str.replace(',', '.').astype(float) applied before concatenation.",
    },
    {
        "Dataset": "healthy_discovery.csv",
        "Column": "original_glszm_ZonePercentage",
        "Issue": "Same locale issue: '0,00001' stored as string.",
        "Impact": "Column parsed as str.",
        "Pipeline fix": "Cell 2: same comma-to-dot replacement.",
    },
    {
        "Dataset": "healthy_discovery.csv · healthy_signa.csv",
        "Column": "CIRRHOSIS_TYPE · GRADE_HE · MELD",
        "Issue": "Columns are NaN for all rows (not applicable in healthy subjects).",
        "Impact": "dtype inferred as float64 for GRADE_HE and MELD; object/float for CIRRHOSIS_TYPE.",
        "Pipeline fix": "By design; healthy control files are never passed to outcome model cells.",
    },
    {
        "Dataset": "radiomic_HE_dataset_extA.csv · radiomic_HE_dataset_extB.csv",
        "Column": "MR_System",
        "Issue": (
            "Single unique value per file: 'Signa' in Ext-A, 'Achieva' in Ext-B. "
            "pycombat 0.20 transform() requires all batch IDs seen at fit time to be present."
        ),
        "Impact": "Cannot call standard .transform(); single-scanner batches would be rejected.",
        "Pipeline fix": (
            "Cell 4 (ComBat): reference-based manual transform applied — "
            "Ext-A harmonised against internal 'Signa' batch parameters; "
            "Ext-B against internal 'Discovery' batch parameters."
        ),
    },
]

display(Markdown("### Known Data Quality Flags"))
display(render_dict(quality_flags))

### Known Data Quality Flags

,Dataset,Column,Issue,Impact,Pipeline fix
0,healthy_discovery.csv,original_glrlm_ShortRunEmphasis,"Locale-specific decimal separator: values stored as '0,000004' (comma) instead of 0.000004 (dot).",Column parsed as str instead of float64 by pandas default reader.,"Cell 2 (data loading): str.replace(',', '.').astype(float) applied before concatenation."
1,healthy_discovery.csv,original_glszm_ZonePercentage,"Same locale issue: '0,00001' stored as string.",Column parsed as str.,Cell 2: same comma-to-dot replacement.
2,healthy_discovery.csv · healthy_signa.csv,CIRRHOSIS_TYPE · GRADE_HE · MELD,Columns are NaN for all rows (not applicable in healthy subjects).,dtype inferred as float64 for GRADE_HE and MELD; object/float for CIRRHOSIS_TYPE.,By design; healthy control files are never passed to outcome model cells.
3,radiomic_HE_dataset_extA.csv · radiomic_HE_dataset_extB.csv,MR_System,"Single unique value per file: 'Signa' in Ext-A, 'Achieva' in Ext-B. pycombat 0.20 transform() requires all batch IDs seen at fit time to be present.",Cannot call standard .transform(); single-scanner batches would be rejected.,Cell 4 (ComBat): reference-based manual transform applied — Ext-A harmonised against internal 'Signa' batch parameters; Ext-B against internal 'Discovery' batch parameters.


---
## Cell 10 — Data Directory Setup and Empty Template Generator

This cell creates the `./data/` subdirectory and generates one empty CSV template per dataset (header row only, zero data rows). File names match the production names expected by the pipeline.

---

### ▶ Operational Instructions — 3 Steps to Run the Pipeline

**Step 1 — Generate the empty dataset files**  
Run this cell (Cell 10). Five empty CSV files will be created in `./data/`:
```
data/
├── radiomic_HE_dataset.csv          ← internal cohort
├── radiomic_HE_dataset_extA.csv     ← external validation cohort A
├── radiomic_HE_dataset_extB.csv     ← external validation cohort B
├── healthy_discovery.csv            ← healthy controls (Discovery scanner)
└── healthy_signa.csv                ← healthy controls (Signa scanner)
```
Each file contains the correct 39-column header and no data rows.

**Step 2 — Populate the datasets with your own data**  
Open each CSV file in `./data/` with your preferred tool (Excel, LibreOffice, Python, R, etc.) and fill in your patient/subject records row by row, following the column schema documented in Cells 2–7 of this notebook. Respect the allowed values and data types for each variable. Do **not** rename or reorder columns. Once populated, optionally run Cell 11 to validate the schema before proceeding.

**Step 3 — Run the radiomic pipeline**  
Open `radiomic_pipeline_v1.ipynb` in the same working directory and execute all cells in order (Cells 0–27). The pipeline reads all five CSV files from `./data/`, applies ComBat harmonisation, fits the LASSO radiomic models, and produces all figures, tables, and reproducibility outputs.

---

> **No patient data is generated or exported by this cell.**

In [10]:
import os
from pathlib import Path

# ── Working directory and data directory ─────────────────────────────────────
# WORKING_DIR: root of the repository / notebook working folder.
# DATA_DIR:    subdirectory 'data/' inside WORKING_DIR where all CSV files
#              (both templates and real datasets) must be placed.
#
# By default both are resolved relative to the notebook's own location so that
# the notebook is portable across machines (no hard-coded absolute paths).
# Override WORKING_DIR below if you run the notebook from a different directory.

WORKING_DIR = Path.cwd()           # = directory from which Jupyter was launched
DATA_DIR    = WORKING_DIR / "data" # all CSV files live here

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Working directory : {WORKING_DIR}")
print(f"Data directory    : {DATA_DIR}")

# ── Shared column schema ──────────────────────────────────────────────────────
# All five datasets share this exact column order (39 columns total).
# Column order must be preserved for downstream pipeline compatibility.

SHARED_COLUMNS = [
    # Block 1 — Clinical & demographic (9 columns)
    "MR_System",           # Scanner label (str): 'Discovery' | 'Signa' | 'Achieva'
    "SEX",                 # Biological sex (str): 'M' | 'W'
    "AGE",                 # Age in years (int)
    "CIRRHOSIS_TYPE",      # Aetiology (str | NaN): 'ALD' | 'Viral hepatitis' | 'NASH' | 'Mixed'
    "GRADE_HE",            # West Haven grade (float | NaN): 0.0 | 1.0 | 2.0
    "HE_None",             # Binary indicator: 1 if GRADE_HE == 0 (float)
    "HE_Mild",             # Binary indicator: 1 if GRADE_HE == 1 (float)
    "HE_ModerateSevere",   # Binary indicator: 1 if GRADE_HE == 2 (float)
    "MELD",                # MELD score (int | NaN)
    # Block 2 — Radiomic features — First-Order Statistics (4 columns)
    "original_firstorder_Entropy",
    "original_firstorder_Kurtosis",
    "original_firstorder_Skewness",
    "original_firstorder_Uniformity",
    # Block 2 — Radiomic features — GLCM (1 column)
    "original_glcm_Entropy",
    # Block 2 — Radiomic features — GLDM (3 columns)
    "original_gldm_Busyness",
    "original_gldm_Coarseness",
    "original_gldm_Contrast",
    # Block 2 — Radiomic features — GLRLM (11 columns)
    "original_glrlm_GrayLevelNonUniformity",
    "original_glrlm_HighGrayLevelRunEmphasis",
    "original_glrlm_LongRunEmphasis",
    "original_glrlm_LongRunHighGrayLevelEmphasis",
    "original_glrlm_LongRunLowGrayLevelEmphasis",
    "original_glrlm_LowGrayLevelRunEmphasis",
    "original_glrlm_RunLengthNonUniformity",
    "original_glrlm_RunPercentage",
    "original_glrlm_ShortRunEmphasis",
    "original_glrlm_ShortRunHighGrayLevelEmphasis",
    "original_glrlm_ShortRunLowGrayLevelEmphasis",
    # Block 2 — Radiomic features — GLSZM (10 columns)
    "original_glszm_GrayLevelNonUniformity",
    "original_glszm_HighGrayLevelZoneEmphasis",
    "original_glszm_LargeAreaEmphasis",
    "original_glszm_LargeAreaHighGrayLevelEmphasis",
    "original_glszm_LargeAreaLowGrayLevelEmphasis",
    "original_glszm_LowGrayLevelZoneEmphasis",
    "original_glszm_SmallAreaEmphasis",
    "original_glszm_SmallAreaHighGrayLevelEmphasis",
    "original_glszm_SmallAreaLowGrayLevelEmphasis",
    "original_glszm_ZonePercentage",
    "original_glszm_ZoneSizeNonUniformity",
]

# Verify expected column count
assert len(SHARED_COLUMNS) == 39, (
    f"Schema integrity check failed: expected 39 columns, got {len(SHARED_COLUMNS)}."
)
print(f"Schema integrity OK: {len(SHARED_COLUMNS)} columns defined.")

# ── Dataset names ─────────────────────────────────────────────────────────────
DATASETS = [
    "radiomic_HE_dataset",
    "radiomic_HE_dataset_extA",
    "radiomic_HE_dataset_extB",
    "healthy_discovery",
    "healthy_signa",
]

# ── Generate empty template CSVs inside DATA_DIR ──────────────────────────────
# Each template file contains the header row only (zero data rows).
# File names match the production CSV names exactly so that the pipeline
# Cell 2 (data loading) can be pointed at DATA_DIR without any renaming.
#
# To populate a template with real data, open the CSV in any spreadsheet
# application or text editor and append rows following the column schema
# documented in Cells 2–7 of this notebook.

print(f"\nWriting empty CSV templates to: {DATA_DIR}\n")

for name in DATASETS:
    out_path = DATA_DIR / f"{name}.csv"
    # Zero-row DataFrame with the full 39-column schema
    pd.DataFrame(columns=SHARED_COLUMNS).to_csv(out_path, index=False)
    print(f"  ✓  {out_path.relative_to(WORKING_DIR)}")

print("\nAll templates generated. No patient data is contained in these files.")
print("Place your actual CSV datasets in the same DATA_DIR before running the pipeline.")

Working directory : d:\Radiomic_LT_GitHub\Radiomic_HE_GitHub_LT
Data directory    : d:\Radiomic_LT_GitHub\Radiomic_HE_GitHub_LT\data
Schema integrity OK: 39 columns defined.

Writing empty CSV templates to: d:\Radiomic_LT_GitHub\Radiomic_HE_GitHub_LT\data

  ✓  data\radiomic_HE_dataset.csv
  ✓  data\radiomic_HE_dataset_extA.csv
  ✓  data\radiomic_HE_dataset_extB.csv
  ✓  data\healthy_discovery.csv
  ✓  data\healthy_signa.csv

All templates generated. No patient data is contained in these files.
Place your actual CSV datasets in the same DATA_DIR before running the pipeline.


---
## Cell 11 — Schema Validation Against CSV Files in `./data/`

Run this cell to validate that the CSV files in `DATA_DIR` (defined in Cell 10 as `./data/`) conform to the schema documented in this notebook.  
Cell 10 must be executed first so that `DATA_DIR`, `SHARED_COLUMNS`, and `DATASETS` are available in the kernel namespace.

Status codes:
- **OK** — file present and columns match schema exactly (correct names and order)
- **MISSING** — file not found in `DATA_DIR`
- **MISMATCH** — file found but columns differ from schema (details reported)

In [11]:
# ── Schema Validation ─────────────────────────────────────────────────────────
# This cell compares the SHARED_COLUMNS list (defined in Cell 10) against
# the actual CSV headers found in DATA_DIR (also defined in Cell 10).
# Any discrepancy — missing column, extra column, or wrong order — is reported.
#
# Prerequisites: run Cell 10 first so that WORKING_DIR, DATA_DIR,
#                SHARED_COLUMNS, and DATASETS are defined.
#
# Usage:
#   1. Copy your real CSV files into DATA_DIR (./data/ by default).
#   2. Run this cell. All five files must return Status = OK.

print(f"Validating CSV files in: {DATA_DIR}\n")

validation_results = []

for name in DATASETS:
    fpath = DATA_DIR / f"{name}.csv"
    if not fpath.is_file():
        validation_results.append(
            {
                "File": f"{name}.csv",
                "Status": "MISSING",
                "Details": f"File not found at {fpath}",
            }
        )
        continue

    # Read header only (nrows=0) — no patient data loaded
    actual_cols = list(pd.read_csv(fpath, nrows=0).columns)

    extra    = [c for c in actual_cols   if c not in SHARED_COLUMNS]
    missing  = [c for c in SHARED_COLUMNS if c not in actual_cols]
    order_ok = actual_cols == SHARED_COLUMNS

    if not extra and not missing and order_ok:
        status  = "OK"
        details = f"{len(actual_cols)} columns match schema exactly."
    else:
        status = "MISMATCH"
        parts  = []
        if missing:
            parts.append(f"Missing columns: {missing}")
        if extra:
            parts.append(f"Extra columns: {extra}")
        if not order_ok and not missing and not extra:
            parts.append("Column order differs from schema.")
        details = " | ".join(parts)

    validation_results.append(
        {"File": f"{name}.csv", "Status": status, "Details": details}
    )

display(Markdown("### Schema Validation Results"))
display(render_dict(validation_results))

Validating CSV files in: d:\Radiomic_LT_GitHub\Radiomic_HE_GitHub_LT\data



### Schema Validation Results

,File,Status,Details
0,radiomic_HE_dataset.csv,OK,39 columns match schema exactly.
1,radiomic_HE_dataset_extA.csv,OK,39 columns match schema exactly.
2,radiomic_HE_dataset_extB.csv,OK,39 columns match schema exactly.
3,healthy_discovery.csv,OK,39 columns match schema exactly.
4,healthy_signa.csv,OK,39 columns match schema exactly.


---
## Citation and Licence

If you use these datasets or this pipeline, please cite:

> **[Author list blinded for review].** Non-invasive grading of hepatic encephalopathy with globus pallidus T1-weighted MRI radiomics: a multicentre, multi-scanner, multi-vendor study. *Liver Transplantation* (under review).

Radiomic feature definitions follow:  
> Zwanenburg A et al. The Image Biomarker Standardization Initiative. *Radiology* 2020;295:328–338. DOI: 10.1148/radiol.2020191145

Reporting follows:  
> Collins GS et al. TRIPOD+AI statement. *BMJ* 2024;385:e078378.

**Licence:** Creative Commons Attribution 4.0 International (CC BY 4.0).  
Data are provided for research reproducibility purposes only. Clinical decisions must not be based on these data alone.

---
*Last updated: 2026 — pipeline version v1.0*